### Phase 2: Feature Engineering

In [1]:
# =====================================================
# 🧠 PHASE 2 — FEATURE ENGINEERING
# =====================================================

import os
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

%load_ext nb_black

<IPython.core.display.Javascript object>

In [2]:
# -----------------------------------------------------
# Step 1: Setup paths & logging
# -----------------------------------------------------

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
CONCALL_DIR = PROJECT_ROOT / "data" / "concall"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.WARNING)

print(f"📈 Found {len(list(DATA_DIR.glob('*.csv')))} raw files.")

📈 Found 100 raw files.


<IPython.core.display.Javascript object>

In [3]:
# -----------------------------------------------------
# Step 2: Technical Indicator Functions
# -----------------------------------------------------

def compute_technical_features(df):

    df = df.sort_values("Date").copy()
    
    # Returns
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_5d"] = df["Close"].pct_change(5)
    df["ret_21d"] = df["Close"].pct_change(21)

    # Momentum
    df["momentum_1m"] = df["Close"].pct_change(21)
    df["momentum_3m"] = df["Close"].pct_change(63)
    df["momentum_6m"] = df["Close"].pct_change(126)

    # Volatility
    df["vol_30d"] = df["ret_1d"].rolling(30).std().shift(1)
    df["vol_90d"] = df["ret_1d"].rolling(90).std().shift(1)

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["rsi_14"] = (100 - (100 / (1 + rs))).shift(1)

    # MACD
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()

    df["macd"] = (ema12 - ema26).shift(1)
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean().shift(1)
    df["macd_hist"] = (df["macd"] - df["macd_signal"]).shift(1)

    # EMA Cross
    df["ema_fast"] = df["Close"].ewm(span=20, adjust=False).mean()
    df["ema_slow"] = df["Close"].ewm(span=50, adjust=False).mean()

    df["ema_cross"] = (df["ema_fast"] > df["ema_slow"]).astype(int).shift(1)

    # Volume Features
    df["volume_change"] = df["Volume"].pct_change().shift(1)

    vol_mean = df["Volume"].rolling(30).mean()
    vol_std = df["Volume"].rolling(30).std()

    df["volume_zscore"] = ((df["Volume"] - vol_mean) / vol_std).shift(1)

    df["volume_momentum"] = (df["Volume"] / df["Volume"].rolling(21).mean()).shift(1)

    # Price Range Position
    rolling_high = df["High"].rolling(20).max()
    rolling_low = df["Low"].rolling(20).min()

    df["range_position"] = ((df["Close"] - rolling_low) /
                           (rolling_high - rolling_low)).shift(1)

    # Distribution Shape
    df["ret_skew_10"] = df["ret_1d"].rolling(10).skew().shift(1)
    df["ret_kurt_10"] = df["ret_1d"].rolling(10).kurt().shift(1)
    df["ret_skew_30"] = df["ret_1d"].rolling(30).skew().shift(1)
    df["ret_kurt_30"] = df["ret_1d"].rolling(30).kurt().shift(1)

    return df


<IPython.core.display.Javascript object>

### Engineered Feature Descriptions

#### Trading Window Assumptions
- Quant finance commonly assumes **21 trading days per month**.
- Standard rolling windows used:
  - **21 days** → ~1 month
  - **63 days** → ~3 months
  - **126 days** → ~6 months

#### Leakage Prevention
- `shift(1)` is applied across relevant features to prevent **look-ahead bias**.
- Without shifting, the present day value could leak into the feature calculation.

#### Volatility & Portfolio Metrics
- Volatility is included as a feature.
- **Sharpe Ratio** and **Sortino Ratio** are portfolio-level metrics, so they will be calculated later using volatility in **Phase 4 – Portfolio Construction**.

#### Momentum Indicators

**RSI (Relative Strength Index)**
- Default RSI uses the **last 14 trading days**.
- Two intermediate features are created:
  - **gain**
  - **loss**
- Example transformation:

  Returns: `-1, -3, 5, 2`  
  Loss column: `-1, -3, 0, 0`  
  Gain column: `0, 0, 5, 2`

- RSI helps identify **overbought and oversold conditions** in stocks.

**MACD (Moving Average Convergence Divergence)**
- Formula: **MACD = EMA12 − EMA26** (short-term EMA minus long-term EMA).
- Captures **monthly trend acceleration**.
- The **12 and 26 trading day EMAs** follow the standard specification introduced by Gerald Appel.
- Interpretation:
  - **MACD > 0** → bullish momentum
  - **MACD < 0** → bearish momentum

#### Volume-Based Features
- **volume_z**
  - Measures unusual trading activity.
  - Example:  
    - `volume_z = 3` → unusually high trading activity  
    - `volume_z = 0` → normal activity

- **volume_momentum**
  - Measures recent **liquidity surge**.
  - High values indicate **increased market attention**.

#### Price Range Position
- **Range position**
  - Indicates where the price lies within its recent range.
  - `0` → near support  
  - `1` → near resistance

#### Distribution Shape of Returns

- **Rolling Skewness**
  - Measures asymmetry of returns.
  - Positive skew → small losses with occasional large gains.
  - Negative skew → frequent small gains with risk of sharp crashes.

- **Rolling Kurtosis**
  - Measures likelihood of **extreme returns**.
  - Example situations:
    - calm trading periods followed by sudden large moves due to earnings surprises
    - panic-driven market reactions such as geopolitical events (e.g., Iran–Israel conflict).

In [4]:
# -----------------------------------------------------
# Step 2.5: Enhanced Concall Tone + Guidance Features
# -----------------------------------------------------

# --- Financial Tone Dictionaries (Unchanged) ---

POSITIVE_WORDS = set(
    [
        "growth",
        "strong",
        "improve",
        "improved",
        "improving",
        "profit",
        "profits",
        "profitable",
        "expansion",
        "expand",
        "expanding",
        "robust",
        "positive",
        "optimistic",
        "confidence",
        "confident",
        "opportunity",
        "opportunities",
        "momentum",
        "resilient",
        "outperform",
        "record",
        "efficient",
        "efficiency",
    ]
)

NEGATIVE_WORDS = set(
    [
        "decline",
        "declining",
        "declined",
        "loss",
        "losses",
        "weak",
        "weakness",
        "risk",
        "risks",
        "slowdown",
        "pressure",
        "pressures",
        "uncertain",
        "uncertainty",
        "challenge",
        "challenges",
        "volatile",
        "volatility",
        "concern",
        "concerns",
        "drop",
        "decrease",
        "decreasing",
        "headwind",
        "headwinds",
        "underperform",
        "challenging",
        "negative",
    ]
)

GUIDANCE_WORDS = set(
    [
        "guidance",
        "outlook",
        "forecast",
        "expect",
        "expects",
        "expected",
        "projection",
        "projections",
        "estimate",
        "estimates",
        "target",
        "targets",
        "aim",
        "anticipate",
        "anticipates",
    ]
)


def compute_concall_tone(text):
    """Compute tone and guidance features from transcript text."""

    if pd.isna(text):
        return 0, 0, 0, 0, 0

    words = str(text).lower().split()
    total_words = len(words)

    if total_words == 0:
        return 0, 0, 0, 0, 0

    pos_count = sum(1 for w in words if w in POSITIVE_WORDS)
    neg_count = sum(1 for w in words if w in NEGATIVE_WORDS)
    guidance_flag = int(any(w in GUIDANCE_WORDS for w in words))

    pos_ratio = pos_count / total_words
    neg_ratio = neg_count / total_words
    net_tone = pos_ratio - neg_ratio
    tone_intensity = pos_ratio + neg_ratio

    return pos_ratio, neg_ratio, net_tone, tone_intensity, guidance_flag


def load_and_aggregate_concall(symbol):
    concall_file = CONCALL_DIR / f"{symbol}_concall.csv"

    if not concall_file.exists():
        return None

    df = pd.read_csv(concall_file)

    if df.empty:
        return None

    # ✅ Standardize column name to "Date"
    df["Date"] = pd.to_datetime(df["date"], errors="coerce")
    df.dropna(subset=["Date"], inplace=True)

    if "text" not in df.columns:
        return None

    # --- Compute tone metrics ---
    tone_features = df["text"].apply(compute_concall_tone)

    df[
        ["pos_ratio", "neg_ratio", "net_tone", "tone_intensity", "guidance_flag"]
    ] = pd.DataFrame(tone_features.tolist(), index=df.index)

    df = df.sort_values("Date")

    # 🔴 CRITICAL: Shift BEFORE rolling
    # Calculate the difference between the tone of rolling mean of previous 4 concalls and current concall
    df["tone_rolling_mean"] = df["net_tone"].shift(1).rolling(4, min_periods=1).mean()

    df["tone_surprise"] = df["net_tone"] - df["tone_rolling_mean"]

    # --- Aggregate Daily ---
    concall_agg = (
        df.groupby("Date")
        .agg(
            concall_count=("text", "count"),
            concall_pos_ratio=("pos_ratio", "mean"),
            concall_neg_ratio=("neg_ratio", "mean"),
            concall_net_tone=("net_tone", "mean"),
            concall_tone_intensity=("tone_intensity", "mean"),
            concall_tone_surprise=("tone_surprise", "mean"),
            concall_guidance_flag=("guidance_flag", "max"),
        )
        .reset_index()
    )

    return concall_agg

<IPython.core.display.Javascript object>

In [5]:
# -----------------------------------------------------
# Step 3: Process Each Ticker (CORRECTED FINAL VERSION)
# -----------------------------------------------------

all_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
print(f"📈 Processing {len(all_files)} tickers...")

for file in tqdm(all_files):
    try:
        file_path = DATA_DIR / file
        symbol = file.replace(".csv", "")

        df = pd.read_csv(file_path)

        # --- Clean column names ---
        df.columns = [c.strip().lower() for c in df.columns]

        col_map = {
            "date": "Date",
            "openprice": "Open",
            "open": "Open",
            "highprice": "High",
            "high": "High",
            "lowprice": "Low",
            "low": "Low",
            "closeprice": "Close",
            "close": "Close",
            "totaltradedquantity": "Volume",
            "volume": "Volume",
        }

        df.rename(columns=col_map, inplace=True)

        # --- Validate Required Columns ---
        required_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
        for c in required_cols:
            if c not in df.columns:
                raise ValueError(f"{file} missing column: {c}")

        # --- Parse Date ---
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df.dropna(subset=["Date"], inplace=True)

        # --- Numeric Conversion ---
        for col in ["Open", "High", "Low", "Close", "Volume"]:
            df[col] = (
                df[col].astype(str).str.replace(",", "", regex=False).astype(float)
            )

        df.sort_values("Date", inplace=True)

        # =================================================
        # 🔹 Merge Concall Features (ONLY ONCE)
        # =================================================

        concall_df = load_and_aggregate_concall(symbol)

        if concall_df is not None:
            df = df.merge(concall_df, on="Date", how="left")
            # Forward fill concall features
            df[concall_cols] = df[concall_cols].ffill()
            # Fill the beginning (before first call)
            df[concall_cols] = df[concall_cols].fillna(0)

            concall_cols = [
                "concall_count",
                "concall_pos_ratio",
                "concall_neg_ratio",
                "concall_net_tone",
                "concall_tone_intensity",
                "concall_tone_surprise",
                "concall_guidance_flag",
            ]

            df[concall_cols] = df[concall_cols].fillna(0)

        else:
            df[
                [
                    "concall_count",
                    "concall_pos_ratio",
                    "concall_neg_ratio",
                    "concall_net_tone",
                    "concall_tone_intensity",
                    "concall_tone_surprise",
                    "concall_guidance_flag",
                ]
            ] = 0

        # =================================================
        # 🔹 Technical Features
        # =================================================

        df_feat = compute_technical_features(df)

        # --- Drop early NA rows (indicator warmup) ---
        min_required_cols = ["momentum_6m", "vol_90d", "rsi_14", "macd_hist"]
        df_feat.dropna(subset=min_required_cols, inplace=True)

        rows_before = len(df_feat)
        df_feat = df_feat.dropna()
        rows_after = len(df_feat)
        print(f"Rows before dropna: {rows_before}")
        print(f"Rows after dropna: {rows_after}")
        print(f"Rows removed: {rows_before - rows_after}")

        # =================================================
        # 🔹 Save Features
        # =================================================

        df_feat.to_csv(FEATURE_DIR / file, index=False)

    except Exception as e:
        logging.warning(f"❌ Failed for {file}: {e}")
        continue

print("✅ Feature engineering completed for all tickers!")

📈 Processing 100 tickers...


  1%|          | 1/100 [00:00<00:36,  2.70it/s]

Rows before dropna: 1409
Rows after dropna: 1409
Rows removed: 0


  2%|▏         | 2/100 [00:00<00:31,  3.15it/s]

Rows before dropna: 1244
Rows after dropna: 1244
Rows removed: 0


...

 99%|█████████▉| 99/100 [00:27<00:00,  4.30it/s]

Rows before dropna: 1412
Rows after dropna: 1412
Rows removed: 0


100%|██████████| 100/100 [00:27<00:00,  3.62it/s]

Rows before dropna: 1412
Rows after dropna: 1412
Rows removed: 0
✅ Feature engineering completed for all tickers!


<IPython.core.display.Javascript object>

In [ ]:
# -----------------------------------------------------
# Step 4: Sanity Check Summary (UPDATED)
# -----------------------------------------------------

feature_files = list(FEATURE_DIR.glob("*.csv"))

if len(feature_files) == 0:
    print("⚠️ No feature files found in FEATURE_DIR.")
else:
    sample_file = feature_files[0]
    df_sample = pd.read_csv(sample_file)

    print(f"\n🧩 Sample engineered dataset preview: {sample_file.name}")
    print(f"Shape: {df_sample.shape}")
    print(f"Columns: {len(df_sample.columns)}")

    preview_cols = [
        # Price
        "Date",
        "Close",
        # Technical
        "momentum_1m",
        "vol_30d",
        "rsi_14",
        "macd_hist",
        "ema_cross",
        # News
        "sent_3d",
        "news_intensity",
        # Concall (New Tone Features)
        "concall_pos_ratio",
        "concall_neg_ratio",
        "concall_net_tone",
        "concall_tone_surprise",
        "concall_guidance_flag",
    ]

    preview_cols = [c for c in preview_cols if c in df_sample.columns]

    print("\n📊 Preview:")
    print(df_sample.head(10)[preview_cols])

    print("\n🔎 Missing values:")
    print(df_sample[preview_cols].isna().sum())

    print("\n📈 Basic Stats (Tone Features):")
    tone_cols = [
        "concall_pos_ratio",
        "concall_neg_ratio",
        "concall_net_tone",
        "concall_tone_surprise",
    ]
    tone_cols = [c for c in tone_cols if c in df_sample.columns]

    if tone_cols:
        print(df_sample[tone_cols].describe())

In [ ]:
sample_file

In [ ]:
df_sample.head(10)

In [ ]:
# -----------------------------------------------------
# Step 5: Data Availability Check Across All Stocks
# -----------------------------------------------------

feature_files = list(FEATURE_DIR.glob("*.csv"))

availability_summary = []

for file in feature_files:
    df = pd.read_csv(file)

    row_count = len(df)

    # Ensure Date exists and is parsed
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        start_date = df['Date'].min()
        end_date = df['Date'].max()
        date_span_years = (end_date - start_date).days / 365
    else:
        start_date = None
        end_date = None
        date_span_years = None

    availability_summary.append({
        "symbol": file.stem,
        "rows": row_count,
        "start_date": start_date,
        "end_date": end_date,
        "years_span": round(date_span_years, 2) if date_span_years else None
    })

availability_df = pd.DataFrame(availability_summary)


In [ ]:
# -----------------------------------------------------
# Summary Stats
# -----------------------------------------------------

print("\n📊 Data Availability Summary")
print("-" * 40)
print(availability_df['rows'].describe())

print("\n📅 Years Span Summary")
print("-" * 40)
print(availability_df['years_span'].describe())

# -----------------------------------------------------
# Stocks Below 1000 Rows
# -----------------------------------------------------

low_data = availability_df[availability_df['rows'] < 1000]

print(f"\n⚠️ Stocks with < 1000 rows: {len(low_data)}")
print(low_data.sort_values("rows").head(10))


In [ ]:
# -----------------------------------------------------
# Optional: Histogram of Row Distribution
# -----------------------------------------------------
plt.figure(figsize=(10, 6), dpi=300)
plt.hist(availability_df["rows"], bins=20)
plt.title("Distribution of Data Availability (Rows per Stock)")
plt.xlabel("Number of Rows")
plt.ylabel("Number of Stocks")
plt.show()

In [ ]:
# -----------------------------------------------------
# Step 6: Filter Stocks With Minimum 1000 Rows
# -----------------------------------------------------

MIN_DAYS = 1000

feature_files = list(FEATURE_DIR.glob("*.csv"))

kept = []
removed = []

for file in feature_files:
    df = pd.read_csv(file)

    if len(df) >= MIN_DAYS:
        kept.append(file.stem)
    else:
        removed.append(file.stem)
        file.unlink()  # delete file

print("\n🗑 Removed Stocks (< 1000 rows):", len(removed))
print("Sample removed:", removed[:10])

print("\n✅ Stocks Remaining:", len(kept))

In [ ]:
# -----------------------------------------------------
# Final Dataset Verification
# -----------------------------------------------------

remaining_files = list(FEATURE_DIR.glob("*.csv"))

row_counts = []

for file in remaining_files:
    df = pd.read_csv(file)
    row_counts.append(len(df))

print("\n📊 Remaining Dataset Stats")
print("Stocks:", len(remaining_files))
print("Min rows:", min(row_counts))
print("Median rows:", np.median(row_counts))
print("Max rows:", max(row_counts))

In [ ]:
print(
    "Total rows across all stocks:",
    sum(len(pd.read_csv(f)) for f in FEATURE_DIR.glob("*.csv")),
)